In [1]:
import os
import shutil
import sys
from pathlib import Path

In [ ]:
# ========================================================
# 0. CẤU HÌNH THƯ MỤC CODE & DỮ LIỆU
# ========================================================
is_kaggle = os.path.exists("/kaggle/working")

if is_kaggle:
    # Nếu chạy trên Kaggle, add thư mục chứa code vào sys.path
    # TODO: Đổi tên dataset thư mục code của bạn trên Kaggle nếu cần
    code_dir_candidates = [
        Path("/kaggle/input/mid-fusion"),
    ]
    code_dir = next(
        (c for c in code_dir_candidates if (c / "config.py").exists()), None
    )
    if code_dir:
        sys.path.insert(0, str(code_dir))
        print(f"[+] Đã trỏ sys.path tới thư mục code: {code_dir}")
    else:
        print("[!] CẢNH BÁO: Không tìm thấy thư mục chứa code (config.py).")

[+] Đã trỏ sys.path tới thư mục code: /kaggle/input/datasets/trnhintmh03/isic-04-07


In [3]:
from config import MidFusionLiteConfig, build_ablation_columns
from runner import run_mid_fusion_cv


def resolve_kaggle_data_paths(config):
    if is_kaggle:
        print("\n[+] Đang tự động tìm kiếm bộ dữ liệu ISIC 2024 trên Kaggle...")
        dataset_root = "/kaggle/input/competitions/isic-2024-challenge"

        # Quét toàn bộ thư mục /kaggle/input để tìm file train-metadata.csv
        for root, dirs, files in os.walk("/kaggle/input"):
            if "train-metadata.csv" in files:
                dataset_root = Path(root)
                break

        if dataset_root:
            print(f"    -> Đã tìm thấy dữ liệu ISIC tại: {dataset_root}")
            return config.with_updates(
                train_metadata_path=str(dataset_root / "train-metadata.csv"),
                test_metadata_path=str(dataset_root / "test-metadata.csv"),
                sample_submission_path=str(dataset_root / "sample_submission.csv"),
                train_hdf5_path=str(dataset_root / "train-image.hdf5"),
                test_hdf5_path=str(dataset_root / "test-image.hdf5"),
            )
        else:
            print(
                "    [!] LỖI NGHIÊM TRỌNG: Không tìm thấy file train-metadata.csv trong /kaggle/input/"
            )
            print(
                "    [!] Vui lòng đảm bảo bạn đã 'Add Data' bộ dataset của cuộc thi ISIC 2024 vào notebook!"
            )
    return config

In [ ]:
is_kaggle = os.path.exists("/kaggle/working")

# Thư mục chứa tabm_fold_0.pt, tabm_fold_1.pt, ...
if is_kaggle:
    tabm_weights = "/kaggle/input/datasets/khoangdz/isic-2024-tabm-backbone"

# ========================================================
# 2. CẤU HÌNH MID-FUSION LITE (STAGE 3)
# ========================================================
# Thư mục chứa image_only_fold_0.pt, image_only_fold_1.pt, ...
# (checkpoint đầy đủ nhánh ảnh đã train, KHÔNG phải file backbone timm thô).
if is_kaggle:
    image_weights_dir = (
        "/kaggle/input/datasets/khoangdz/isic-2024-effb0-backbone"
    )
    out_dir = "/kaggle/working/fusion_output"

In [ ]:
tabm_numeric_cols, tabm_categorical_cols = build_ablation_columns(
    use_engineered_abcd=True,
    use_patient_norm=True,
    use_ugly_duckling_adv=True,
    use_categorical_context=True,
)

In [7]:
fusion_config = MidFusionLiteConfig(
    preset_name="kaggle_top1_fusion",
    n_splits=5,
    fold_indices=None,  # Đổi thành [0, 1] nếu chỉ muốn chạy test 1-2 fold
    # --- A. CẤU HÌNH IMAGE (TOP-1 BACKBONE) ---
    image_size=224,
    backbone_name="efficientnet_b0",
    # Không cần backbone thô: nhánh ảnh đầy đủ được nạp qua pretrained_image_dir bên dưới.
    backbone_weights_path=None,
    train_image_backbone=True,  # Sẽ unfreeze ở Stage 3B
    download_pretrained_backbone=False,  # Không tải từ net vì đã dùng Top-1 local/Kaggle weights
    # --- B. CẤU HÌNH TABULAR (Phải khớp với file tabular-only-*.ipynb của bạn) ---
    tabm_k=32,
    tabm_n_blocks=2,
    tabm_d_block=256,
    n_bins=48,
    d_embedding=32,
    tab_dropout=0.3,
    # Bật đầy đủ các nhóm feature để FE sinh đúng bộ cột như Stage 1 (Tabular notebook)
    use_contextual_group_norm_features=True,
    use_ugly_duckling_features=True,
    use_anatomical_zscore_features=True,
    use_area_aggregation_features=True,
    # Tập cột đưa vào TabM — khớp tuyệt đối notebook (610 cột sau khi runner lọc).
    numeric_cols=tabm_numeric_cols,
    categorical_cols=tabm_categorical_cols,
    # --- C. CẤU HÌNH FUSION ---
    fusion_mode="cross_attention",
    fusion_hidden_dim=256,
    fusion_dropout=0.25,
    cross_attention_heads=4,
    # 85% both, 7.5% image-only, 7.5% tabular-only during fusion training.
    modality_dropout_prob=0.15,
    # --- D. STAGE 3: nạp hai checkpoint độc lập rồi train lại fusion ---
    run_mode="fusion",
    # Stage 1/2 chỉ load checkpoint theo fold, không train lại hai nhánh độc lập.
    pretrained_tabular_dir=tabm_weights,
    pretrained_image_dir=image_weights_dir,
    stage3_freeze_epochs=3,
    stage3_unfreeze_epochs=5,
    # Tắt auxiliary losses để chỉ tính loss duy nhất sau Fusion
    aux_loss_image_weight=0.0,
    aux_loss_tab_weight=0.0,
    stage3a_lr=1e-4,
    stage3b_lr=1e-5,  # lr nền cho backbone ảnh ở Stage 3B
    # Stage 3B dùng lr phân tầng: backbone = stage3b_lr (1e-5), còn
    # tabular_encoder + fusion_head = stage3b_lr * mult (5e-5) để cổng fusion
    # tiếp tục học thay vì đứng yên. Đặt =1.0 để tắt phân tầng.
    stage3b_head_lr_mult=5.0,
    train_batch_size=64,  # Giảm batch size vì ảnh 336 tốn VRAM
    valid_batch_size=64,
    negative_ratio=5,  # Tỷ lệ undersample cho Fusion (nhánh ảnh không cần quá nhiều âm tính)
    output_dir=out_dir,
)

# Cập nhật đường dẫn data nếu chạy trên Kaggle
fusion_config = resolve_kaggle_data_paths(fusion_config)


[+] Đang tự động tìm kiếm bộ dữ liệu ISIC 2024 trên Kaggle...
    -> Đã tìm thấy dữ liệu ISIC tại: /kaggle/input/competitions/isic-2024-challenge


In [8]:
print("\n[+] Bắt đầu quá trình huấn luyện Stage 3 Fusion...")
results = run_mid_fusion_cv(fusion_config)

# Kaggle yêu cầu file nộp bài phải có tên đúng là "submission.csv" nằm ở thư mục root /kaggle/working/
if is_kaggle:
    src_sub = os.path.join(out_dir, "submission_mid_fusion_lite.csv")
    dst_sub = "/kaggle/working/submission.csv"
    if os.path.exists(src_sub):
        shutil.copy(src_sub, dst_sub)
        print(f"\n[+] Đã copy file submission ra thư mục gốc Kaggle: {dst_sub}")
    else:
        print(
            f"\n[!] WARNING: Không tìm thấy file {src_sub} để tạo submission.csv!"
        )


[+] Bắt đầu quá trình huấn luyện Stage 3 Fusion...
Running preset: kaggle_top1_fusion
Device: CUDA
Output directory: /kaggle/working/fusion_output
Fold column 'fold' not found. Generating fold splits dynamically using StratifiedGroupKFold.
Selected folds: [0, 1, 2, 3, 4]
Fold 1/5


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Numeric features: 610 | Categorical features: 5


/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 333-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


  [end-to-end (both)] sampler 1:5 -> pos_weight_sampled=2.236
Loaded local backbone weights: /kaggle/working/single_backbone/efficientnet_b0_backbone.pth | missing=0 | unexpected=0
Epoch 01 | train_loss=0.70349 | valid_loss=0.10078 | valid_auroc=0.90128 | valid_pauc=0.13483
Epoch 02 | train_loss=0.45660 | valid_loss=0.13435 | valid_auroc=0.93910 | valid_pauc=0.15380
Epoch 03 | train_loss=0.42024 | valid_loss=0.13544 | valid_auroc=0.94459 | valid_pauc=0.15858
Epoch 04 | train_loss=0.37716 | valid_loss=0.09046 | valid_auroc=0.94779 | valid_pauc=0.16094
Epoch 05 | train_loss=0.32654 | valid_loss=0.14251 | valid_auroc=0.95968 | valid_pauc=0.16896
Epoch 06 | train_loss=0.29958 | valid_loss=0.17144 | valid_auroc=0.95863 | valid_pauc=0.16792
Epoch 07 | train_loss=0.27530 | valid_loss=0.09331 | valid_auroc=0.96642 | valid_pauc=0.17399
Epoch 08 | train_loss=0.27142 | valid_loss=0.12507 | valid_auroc=0.96342 | valid_pauc=0.17172
Epoch 09 | train_loss=0.25941 | valid_loss=0.09560 | valid_auroc=0.

/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 333-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 333-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


  [end-to-end (both)] sampler 1:5 -> pos_weight_sampled=2.236
Loaded local backbone weights: /kaggle/working/single_backbone/efficientnet_b0_backbone.pth | missing=0 | unexpected=0
Epoch 01 | train_loss=0.68072 | valid_loss=0.14972 | valid_auroc=0.91878 | valid_pauc=0.13746
Epoch 02 | train_loss=0.49083 | valid_loss=0.13722 | valid_auroc=0.94595 | valid_pauc=0.15392
Epoch 03 | train_loss=0.42406 | valid_loss=0.13730 | valid_auroc=0.95065 | valid_pauc=0.15717
Epoch 04 | train_loss=0.37849 | valid_loss=0.19565 | valid_auroc=0.95825 | valid_pauc=0.16522
Epoch 05 | train_loss=0.34155 | valid_loss=0.13789 | valid_auroc=0.96257 | valid_pauc=0.16925
Epoch 06 | train_loss=0.31090 | valid_loss=0.11619 | valid_auroc=0.96717 | valid_pauc=0.17440
Epoch 07 | train_loss=0.28744 | valid_loss=0.13101 | valid_auroc=0.96021 | valid_pauc=0.16784
Epoch 08 | train_loss=0.27680 | valid_loss=0.13097 | valid_auroc=0.96438 | valid_pauc=0.17010
Early stopping triggered.
Running fold 4 inference (log every 20 st

/usr/local/lib/python3.12/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 333-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(


  [end-to-end (both)] sampler 1:5 -> pos_weight_sampled=2.236
Loaded local backbone weights: /kaggle/working/single_backbone/efficientnet_b0_backbone.pth | missing=0 | unexpected=0
Epoch 01 | train_loss=0.68262 | valid_loss=0.12483 | valid_auroc=0.90573 | valid_pauc=0.13244
Epoch 02 | train_loss=0.47698 | valid_loss=0.16012 | valid_auroc=0.94692 | valid_pauc=0.16046
Epoch 03 | train_loss=0.41782 | valid_loss=0.14230 | valid_auroc=0.96233 | valid_pauc=0.17189
Epoch 04 | train_loss=0.36359 | valid_loss=0.11726 | valid_auroc=0.96986 | valid_pauc=0.17684
Epoch 05 | train_loss=0.33523 | valid_loss=0.08942 | valid_auroc=0.97186 | valid_pauc=0.17820
Epoch 06 | train_loss=0.31064 | valid_loss=0.11131 | valid_auroc=0.97435 | valid_pauc=0.17998
Epoch 07 | train_loss=0.31050 | valid_loss=0.14077 | valid_auroc=0.97335 | valid_pauc=0.17863
Epoch 08 | train_loss=0.29089 | valid_loss=0.15336 | valid_auroc=0.96979 | valid_pauc=0.17492
Early stopping triggered.
Running fold 5 inference (log every 20 st

In [9]:
print("\n[+] Hoàn thành toàn bộ quy trình!")
print("\n" + "=" * 60)
print("  KẾT QUẢ HUẤN LUYỆN FUSION")
print("=" * 60)
cv_auc = results.get('cv_auc')
cv_pauc = results.get('cv_pauc')
print(f"  CV AUC  : {f'{cv_auc:.5f}' if cv_auc is not None else 'N/A'}")
print(f"  CV pAUC : {f'{cv_pauc:.5f}' if cv_pauc is not None else 'N/A'}")
print("=" * 60)


[+] Hoàn thành toàn bộ quy trình!

  KẾT QUẢ HUẤN LUYỆN FUSION
  CV AUC  : N/A
  CV pAUC : 0.17474
